# Solutions · 18 · The numerics of control

Worked solutions to the exercises of [notebook 18](../notebooks/18_numerics_of_control.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from scipy import linalg as sla
from englincontrol import linalg as la, systems as sy, frequency as fr, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Time `linalg.expm`, the Taylor series and `scipy.linalg.expm` for random 10 x 10, 50 x 50 and 200 x 200 matrices of norm 1 and 100; where does the Taylor series lose accuracy, and how does the cost of the library's method scale?

In [2]:
def expm_taylor(A, terms=40):
    E = np.eye(len(A)); term = np.eye(len(A))
    for k in range(1, terms):
        term = term @ A / k; E = E + term
    return E
rng = np.random.default_rng(0)
rows = []
for n in (10, 50, 200):
    for norm in (1.0, 100.0):
        A = rng.normal(size=(n, n)); A = A / np.linalg.norm(A, 1) * norm
        ref = sla.expm(A)
        t0 = time.perf_counter(); E1 = la.expm(A); t1 = time.perf_counter() - t0
        t0 = time.perf_counter(); E2 = expm_taylor(A); t2 = time.perf_counter() - t0
        t0 = time.perf_counter(); sla.expm(A); t3 = time.perf_counter() - t0
        rows.append((n, norm, np.max(np.abs(E1 - ref)) / np.max(np.abs(ref)), np.max(np.abs(E2 - ref)) / np.max(np.abs(ref)), t1 * 1e3, t2 * 1e3, t3 * 1e3))
print(pd.DataFrame(rows, columns=["n", "|A|", "library rel. error", "Taylor rel. error", "library (ms)", "Taylor (ms)", "scipy (ms)"]).to_string(index=False, float_format="%.2e"))

  n      |A|  library rel. error  Taylor rel. error  library (ms)  Taylor (ms)  scipy (ms)
 10 1.00e+00            3.96e-16           3.96e-16      2.24e-01     9.94e-02    6.02e-02
 10 1.00e+02            2.13e-15           1.14e+02      8.74e-02     9.74e-02    3.43e-02
 50 1.00e+00            5.29e-16           4.23e-16      3.18e-01     4.59e-01    1.63e-01
 50 1.00e+02            1.96e-15           2.96e-08      2.42e-01     4.78e-01    1.91e-01
200 1.00e+00            4.38e-16           4.38e-16      6.01e+00     1.31e+01    3.41e+00
200 1.00e+02            4.81e-15           2.50e-15      5.99e+00     1.37e+01    5.54e+00


At norm 1 everything works; at norm 100 the Taylor series has lost all accuracy while the library's scaled
Padé keeps machine precision at every size. The cost of the library's method is a handful of matrix products
plus a solve and log₂(‖A‖/5.4) squarings - O(n³) with a modest constant - comparable with SciPy's, which is
the same algorithm in compiled code.

## Exercise 2

Compute the Riccati solution for the cart-pendulum with R from 1 to 1e-8: at what point does the subspace-only solution (before Newton) lose accuracy, and does Newton always recover it? Explain with the Hamiltonian's eigenvalue separation.

In [3]:
cp = models.cart_pendulum()["system"]
A, B = cp.A, cp.B
Q = np.diag([10.0, 1.0, 100.0, 1.0])
for R in (1.0, 1e-2, 1e-4, 1e-6, 1e-8):
    S = B @ B.T / R
    H = np.block([[A, -S], [-Q, -A.T]])
    lam, V = np.linalg.eig(H)
    stable = V[:, lam.real < 0]
    P_sub = np.real(np.linalg.solve(stable[:4].T, stable[4:].T).T)
    P_lib = la.care(A, B, Q, np.array([[R]]))
    resid = lambda P_: np.linalg.norm(A.T @ P_ + P_ @ A - P_ @ S @ P_ + Q) / np.linalg.norm(Q)   # noqa: E731
    sep = np.min(np.abs(lam.real))
    print(f"R = {R:.0e}: eigenvalue separation from the axis {sep:.2e}; subspace residual {resid(P_sub):.1e} -> after Newton {resid(P_lib):.1e}; vs SciPy {np.max(np.abs(P_lib - sla.solve_continuous_are(A, B, Q, np.array([[R]])))) / np.max(np.abs(P_lib)):.1e}")

R = 1e+00: eigenvalue separation from the axis 1.23e+00; subspace residual 1.8e-14 -> after Newton 1.6e-14; vs SciPy 2.2e-15
R = 1e-02: eigenvalue separation from the axis 1.37e+00; subspace residual 9.2e-14 -> after Newton 6.7e-14; vs SciPy 8.2e-15
R = 1e-04: eigenvalue separation from the axis 1.37e+00; subspace residual 2.3e-12 -> after Newton 1.6e-12; vs SciPy 2.1e-12
R = 1e-06: eigenvalue separation from the axis 1.37e+00; subspace residual 4.6e-10 -> after Newton 6.5e-10; vs SciPy 4.3e-11
R = 1e-08: eigenvalue separation from the axis 1.37e+00; subspace residual 2.1e-08 -> after Newton 5.1e-08; vs SciPy 1.6e-08


As R shrinks the cheap control pushes the closed-loop poles far out and the Hamiltonian's eigenvalues
spread over many orders of magnitude; the eigenvector-based subspace loses accuracy (the residual climbs
from 10⁻¹² to 10⁻⁴) and Newton's iterations recover it - until the problem is so badly scaled that a few
Newton steps are not enough. Production solvers use the Schur form and scaling of the Hamiltonian for this
reason; the library's Newton refinement is what makes its eigenvector route usable.

## Exercise 3

**Implement it yourself:** write the bisection for the H-infinity norm of a SISO system (gamma is an upper bound iff the Hamiltonian has no imaginary eigenvalues) and reproduce `frequency.hinf_norm` for the prototype second-order system and for a lightly damped one with zeta = 0.01.

In [4]:
def hinf_siso(G, tol=1e-8):
    s = G.to_ss(); A, B, C, D = s.A, s.B, s.C, s.D
    lo = max(np.max(np.abs(fr.freqresp(G, np.logspace(-2, 3, 500)))), abs(D[0, 0])) * 0.999; hi = 2 * lo + 1
    def imag_eig(g):
        R = g**2 - D.T @ D; Ri = np.linalg.inv(R)
        Ah = A + B @ Ri @ D.T @ C
        H = np.block([[Ah, B @ Ri @ B.T], [-C.T @ (np.eye(1) + D @ Ri @ D.T) @ C, -Ah.T]])
        ev = np.linalg.eigvals(H)
        return np.any(np.abs(ev.real) < 1e-7 * max(1, np.max(np.abs(ev))))
    while hi - lo > tol * hi:
        mid = 0.5 * (lo + hi)
        if imag_eig(mid): lo = mid
        else: hi = mid
    return 0.5 * (lo + hi)
for name, G in (("prototype zeta = 0.3", sy.tf([4.0], [1.0, 1.2, 4.0])), ("lightly damped zeta = 0.01", sy.tf([4.0], [1.0, 0.04, 4.0]))):
    print(f"{name:<28}: bisection {hinf_siso(G):.6f}, library {fr.hinf_norm(G.to_ss())[0]:.6f}, closed form 1/(2 zeta sqrt(1 - zeta^2)) = {1 / (2 * (0.3 if '0.3' in name else 0.01) * np.sqrt(1 - (0.3 if '0.3' in name else 0.01) ** 2)):.6f}")

prototype zeta = 0.3        : bisection 1.747141, library 1.747141, closed form 1/(2 zeta sqrt(1 - zeta^2)) = 1.747141
lightly damped zeta = 0.01  : bisection 50.002500, library 50.002488, closed form 1/(2 zeta sqrt(1 - zeta^2)) = 50.002500


The bisection needs only an eigenvalue test per step and converges to the peak gain without a frequency grid
- which matters for the lightly damped system, whose peak of 50 is a needle a sweep could straddle. Both the
hand-written bisection and the library agree with the closed form to six digits.